# Tema 1 — Introducción a spaCy y embeddings

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ecamposv/nlp-vision/blob/main/semana-01/Tema-01/EjemploTema1.ipynb)

Puedes ejecutar este notebook localmente (VS Code / Jupyter) o en **Google Colab**.

Si lo abres en Colab, ejecuta primero la celda **Setup para Google Colab** que aparece abajo. En entornos locales esa celda no hace nada y puedes saltarla.


In [ ]:
# === Setup para Google Colab ===
# Esta celda solo hace algo cuando el notebook se ejecuta en Google Colab.
# En entornos locales (VS Code / Jupyter) se ignora y puede saltarse.
import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Colab ya trae spaCy preinstalado; solo descargamos el modelo en español.
    !python -m spacy download es_core_news_md -q
    print("Setup de Colab completado.")
else:
    print("Entorno local detectado, no se requiere setup adicional.")


In [1]:
import spacy
from unidecode import unidecode
import numpy as np
nlp = spacy.load("es_core_news_md")  # modelo español con vectores
texto = ("Muchos años después, frente al pelotón de fusilamiento, el coronel Aureliano Buendía "
         "había de recordar aquella tarde remota en que su padre lo llevó a conocer el hielo.")

doc = nlp(texto)

In [2]:
# Negaciones que NO queremos eliminar aunque spaCy las marque como stopwords
neg = {"no", "ni", "tampoco"}

# Pipeline compacto:
# - tokenización (spaCy)
# - eliminar espacios/puntuación
# - stopwords (conservando negaciones)
# - lematización
# - minúsculas + quitar acentos (unidecode)
tokens_limpios = [
    unidecode(t.lemma_.lower())
    for t in doc
    if not (t.is_space or t.is_punct) and (not t.is_stop or t.lemma_.lower() in neg)
]

print("Tokens finales:", tokens_limpios)


Tokens finales: ['ano', 'frente', 'peloton', 'fusilamiento', 'coronel', 'aureliano', 'buendia', 'recordar', 'remoto', 'padre', 'llevar', 'hielo']


In [3]:
# Vector del documento: usamos los vectores de spaCy sobre el texto limpio
vec = nlp(" ".join(tokens_limpios)).vector
print("Dimensión del vector:", len(vec))
print("Primeras 10 dimensiones:", [round(float(x), 4) for x in vec[:10]])

Dimensión del vector: 300
Primeras 10 dimensiones: [1.4439, 0.4282, 0.0536, -0.148, 0.1665, 0.2844, -0.2723, 0.9998, 0.3741, 1.5903]


In [4]:
pos_lex, neg_lex = {"bueno","excelente","feliz","alegre","genial"}, {"malo","triste","odio","horrible","fusilamiento"}
pos_cent = np.mean([nlp(w).vector for w in pos_lex], axis=0)
neg_cent = np.mean([nlp(w).vector for w in neg_lex], axis=0)
s_pos = (vec @ pos_cent)/(np.linalg.norm(vec)*np.linalg.norm(pos_cent)+1e-9)
s_neg = (vec @ neg_cent)/(np.linalg.norm(vec)*np.linalg.norm(neg_cent)+1e-9)
z = np.array([s_pos, s_neg]); 
z -= z.max()
p = np.exp(z)/np.exp(z).sum()
print(f"Sentimiento (spaCy)  cosPOS={s_pos:.4f} cosNEG={s_neg:.4f}  P(+)={p[0]:.4f} P(-)={p[1]:.4f}  pred={'pos' if p[0]>p[1] else 'neg'}")

Sentimiento (spaCy)  cosPOS=0.1809 cosNEG=0.3705  P(+)=0.4527 P(-)=0.5473  pred=neg
